<a href="https://colab.research.google.com/github/Astralllllll/SemProject_MLEngine/blob/main/KenyaLawCorpus_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Exploration: Kenya Law Corpus
**Project:** LegalMatch AI — Mobile App (Client case-description chatbot + merit-based lawyer matching)
**Dataset:** [MakiniAI/kenya-law-corpus](https://huggingface.co/datasets/MakiniAI/kenya-law-corpus) — 63,603 chunked passages of Kenyan legislation and case law, sourced from new.kenyalaw.org.

**Why this particular dataset?**
 Before matching a client with a lawyer, our matching engine must (a) allow a client to express their legal problem in natural language and (b) route or classify that description toward a practice area, jurisdiction, and precedent context. Instead of depending on generic (non-Kenyan) legal datasets like LEDGAR or ILDC, this corpus provides us with actual Kenyan legal text (case law + legislation) with a `type` field (case_law / legislation) and `court`/`citation` metadata that we can use as candidate output/label variables for a classification or retrieval-based sub-model.

## Setup — load the dataset

In [1]:
!pip install -q datasets pandas

from datasets import load_dataset
import pandas as pd

ds = load_dataset("MakiniAI/kenya-law-corpus")
df = ds["train"].to_pandas()
df.head()

README.md:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

all_chunks.json: reconstructing file:   0%|          |  0.00B / 75.2MB            

all_chunks.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/63603 [00:00<?, ? examples/s]

,chunk_id,text,metadata
0,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Chumba (Traffic Case E827 of 2025) ...,"{""title"":""Republic v Chumba (Traffic Case E827..."
1,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,ed that on 9/05/2025 at about 1100hours along ...,"{""title"":""Republic v Chumba (Traffic Case E827..."
2,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,chool bus without road service license contrar...,"{""title"":""Republic v Chumba (Traffic Case E827..."
3,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,IGNED AND DELIVERED AT NAKURU IN OPEN COURT TH...,"{""title"":""Republic v Chumba (Traffic Case E827..."
4,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Joshua (Criminal Case 765 of 2018) ...,"{""title"":""Republic v Joshua (Criminal Case 765..."


## a. How many rows and columns are contained in the dataset? (3 marks)

In [2]:
n_rows, n_cols = df.shape
print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")
print(f"Column names: {list(df.columns)}")

Rows: 63603
Columns: 3
Column names: ['chunk_id', 'text', 'metadata']


## b. What datatypes are contained in the dataset? (3 marks)

In [3]:
print(df.dtypes)

# 'metadata' is a nested/object column (dict per row) — inspect its inner structure separately
print("\nExample of a 'metadata' entry:")
df["metadata"].iloc[0]

chunk_id    object
text        object
metadata    object
dtype: object

Example of a 'metadata' entry:


'{"title":"Republic v Chumba (Traffic Case E827\\u00a0of\\u00a02025) [2025]\\u00a0KEMC\\u00a094\\u00a0(KLR) (15\\u00a0May\\u00a02025) (Sentence)","url":"https:\\/\\/new.kenyalaw.org\\/akn\\/ke\\/judgment\\/kemc\\/2025\\/94\\/eng@2025-05-15","type":"case_law","chunk_index":0,"total_chunks":4,"citation":"","court":"","date":"May 15, 2025"}'

## c. Is the dataset complete? i.e. no missing values? (2 marks)

In [4]:
missing_top_level = df.isnull().sum()
print("Missing values per top-level column:")
print(missing_top_level)
print(f"\nTotal missing (top-level): {missing_top_level.sum()}")

# The dataset card warns that citation/court/date inside 'metadata' are
# frequently empty for case_law entries, and absent entirely for legislation
# entries. Check that here since it won't show up in the top-level check above.
meta_df = pd.json_normalize(df["metadata"])
print("\nMissing/empty values inside 'metadata' fields:")
print(meta_df.isnull().sum())
print("\n-> Top-level columns (chunk_id, text, metadata) are complete, "
      "but several nested metadata sub-fields (citation, court, date) are "
      "not populated for every row — so the dataset is NOT fully complete "
      "once nested fields are considered.")

Missing values per top-level column:
chunk_id    0
text        0
metadata    0
dtype: int64

Total missing (top-level): 0

Missing/empty values inside 'metadata' fields:
Series([], dtype: float64)

-> Top-level columns (chunk_id, text, metadata) are complete, but several nested metadata sub-fields (citation, court, date) are not populated for every row — so the dataset is NOT fully complete once nested fields are considered.


## d. Slice out the first 15 rows and last 20 rows, merge into `df_sample` (4 marks)

In [5]:
df_sample = pd.concat([df.head(15), df.tail(20)]).reset_index(drop=True)
print(f"df_sample shape: {df_sample.shape}")
df_sample

df_sample shape: (35, 3)


,chunk_id,text,metadata
0,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Chumba (Traffic Case E827 of 2025) ...,"{""title"":""Republic v Chumba (Traffic Case E827..."
1,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,ed that on 9/05/2025 at about 1100hours along ...,"{""title"":""Republic v Chumba (Traffic Case E827..."
2,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,chool bus without road service license contrar...,"{""title"":""Republic v Chumba (Traffic Case E827..."
3,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,IGNED AND DELIVERED AT NAKURU IN OPEN COURT TH...,"{""title"":""Republic v Chumba (Traffic Case E827..."
4,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Joshua (Criminal Case 765 of 2018) ...,"{""title"":""Republic v Joshua (Criminal Case 765..."
5,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,"le two N100 batteries, assorted spanners, one ...","{""title"":""Republic v Joshua (Criminal Case 765..."
6,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,not guilty. The matter was then set down for ...,"{""title"":""Republic v Joshua (Criminal Case 765..."
7,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,d a battery and some spanners were recovered f...,"{""title"":""Republic v Joshua (Criminal Case 765..."
8,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case is defined in the Mozley and Whiteley’s ...,"{""title"":""Republic v Joshua (Criminal Case 765..."
9,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,the prosecution to prove its case beyond reaso...,"{""title"":""Republic v Joshua (Criminal Case 765..."
